# EDA, learning copy

Same cells as `eda.ipynb`. The notes say what each call returns. Run from the repo root. Run the load cell before any later cell.

## Load

`Path` is just the file location. `read_json(..., lines=True)` reads line-delimited JSON: one product per line, not one big list. `df` is the table. `shape` is `(rows, columns)`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

path = Path("data/archive/marketing_sample_for_amazon_com-amazon_fashion_products__20200201_20200430__30k_data.ldjson")
df = pd.read_json(path, lines=True)
df.shape

`columns.tolist()` is the column names in file order.

In [ ]:
df.columns.tolist()

`dtypes` is how pandas stored each column. `object` here means text or a nested dict, not a number yet.

In [ ]:
df.dtypes

`head(3)` is the first three rows, truncated so the table fits. The next cell is one whole record.

In [ ]:
df.head(3)

`iloc[0]` selects the first row by position. `to_dict()` turns that row into a plain dictionary so nested fields such as `sales_rank_in_child_category` are visible.

In [ ]:
df.iloc[0].to_dict()

`isna()` is True on null cells. `sum()` counts those Trues per column. `sort_values(ascending=False)` puts the emptiest column first.

This only sees real nulls. A column can be non-null and still useless, which is what `weight` does with `999999999`.

In [ ]:
df.isna().sum().sort_values(ascending=False)

`nunique()` counts distinct values. `duplicated().sum()` counts rows whose `product_name` already appeared earlier. `uniq_id` should match the row count. A smaller `asin` or `product_name` count means those fields repeat.

In [ ]:
print("uniq_id", df["uniq_id"].nunique())
print("asin", df["asin"].nunique())
print("product_name", df["product_name"].nunique())
df["product_name"].duplicated().sum()

## sales_price

`head(15)` shows the raw strings, including how missing prices look. `value_counts(dropna=False)` counts each raw string and keeps the nulls in the count. `head(15)` on that result is the 15 most common raw values, not the first 15 rows.

In [ ]:
df["sales_price"].head(15)

In [ ]:
df["sales_price"].value_counts(dropna=False).head(15)

`to_numeric(..., errors="coerce")` turns `"590.00"` into `590.0`. A value that is not a number becomes NaN instead of raising. `describe()` then gives count, mean, standard deviation, min, quartiles, and max. Count is how many prices parsed.

In [ ]:
price = pd.to_numeric(df["sales_price"], errors="coerce")
price.describe()

`hist(bins=40)` draws 40 bars of how many products fall in each price range. `price[price > 0]` drops missing prices before `log10`, because the log of NaN is not a bar. The log chart spreads cheap products apart and pulls expensive ones together.

In [ ]:
price.hist(bins=40)
plt.xlabel("sales_price")
plt.show()

np.log10(price[price > 0]).hist(bins=40)
plt.xlabel("log10(sales_price)")
plt.show()

## rating

`head(10)` is the raw rating text. `to_numeric` makes it numeric, `value_counts` counts each rating, and `sort_index()` orders the result from 1.0 up to 5.0 instead of by popularity.

In [ ]:
df["rating"].head(10)

In [ ]:
pd.to_numeric(df["rating"], errors="coerce").value_counts(dropna=False).sort_index()

## weight

`head(15)` is the raw weight text. `value_counts` then shows the most common strings. The top one, `999999999`, is a placeholder stored as text, so `isna()` did not flag it.

In [ ]:
df["weight"].head(15)

In [ ]:
df["weight"].value_counts(dropna=False).head(20)

## brand

`head(15)` shows blanks where brand is missing. `value_counts(dropna=False)` includes that missing count. `nunique(dropna=True)` is how many distinct brand names exist, not counting the nulls.

In [ ]:
df["brand"].head(15)

In [ ]:
df["brand"].value_counts(dropna=False).head(20)

In [ ]:
df["brand"].nunique(dropna=True)

## colour

`dropna().head(20)` skips empty cells and shows the stored strings. They look like `Black|White`, several colours joined by `|`. `value_counts` shows how often each full string repeats. Most strings are rare.

`str.split("|")` breaks each string into a list. `map(len)` is the number of pieces. `value_counts().sort_index()` then says how many products list 2 colours, how many list 3, and so on.

In [ ]:
df["colour"].dropna().head(20)

In [ ]:
df["colour"].value_counts(dropna=False).head(20)

In [ ]:
df["colour"].dropna().str.split("|").map(len).value_counts().sort_index()

## category

`browsenode` is a catalog id. `head(10)` shows the raw ids. `nunique` is how many distinct ids exist.

The rank columns are dictionaries. `head(8)` shows the raw dicts. The key is the category name. The value is the sales rank, which this count ignores.

`map` runs the lambda on every cell. `next(iter(value))` takes the first key, so `{ "MensT_Shirts": "#12151" }` becomes `MensT_Shirts`. An empty or missing cell becomes NaN. `value_counts().head(15)` is the 15 most common names.

In [ ]:
df["browsenode"].head(10)

In [ ]:
df["browsenode"].nunique(dropna=True)

In [ ]:
df["sales_rank_in_child_category"].dropna().head(8)

In [ ]:
df["sales_rank_in_parent_category"].dropna().head(8)

In [ ]:
df["sales_rank_in_child_category"].map(
    lambda value: next(iter(value)) if isinstance(value, dict) and value else np.nan
).value_counts().head(15)

In [ ]:
df["sales_rank_in_parent_category"].map(
    lambda value: next(iter(value)) if isinstance(value, dict) and value else np.nan
).value_counts().head(10)

## name and images

`head(10)` is the raw titles. `str.len()` is the number of characters in each title, and `describe()` summarizes that length.

`medium` holds image URLs. Several URLs can sit in one cell, separated by `|`. `dropna().head(5)` shows five non-empty cells. `isna().sum()` on the three image columns counts how many products have no URL in each field.

In [ ]:
df["product_name"].head(10)

In [ ]:
df["product_name"].str.len().describe()

In [ ]:
df["medium"].dropna().head(5)

In [ ]:
df[["image_urls__small", "medium", "large"]].isna().sum()

## other columns

Each cell is a `value_counts(dropna=False)`. Discount is stored as text such as `50%`, so the count is of those strings, not of numbers. Prime, best-seller, and delivery type are already short labels.

In [ ]:
df["discount_percentage"].value_counts(dropna=False).head(15)

In [ ]:
df["amazon_prime__y_or_n"].value_counts(dropna=False)

In [ ]:
df["best_seller_tag__y_or_n"].value_counts(dropna=False)

In [ ]:
df["delivery_type"].value_counts(dropna=False)